# The Anatomy of AI · Lecture 16 · The Shapes of Superposition — the notebook

Lecture 15 squeezed five ideas into two neurons and got a pentagon. This lecture asked why a pentagon, and what other
shapes there are. Here you rebuild its answers by training the same small networks yourself:

1. the **false echo**: what one feature sends to its neighbours, and how the bias filters it;
2. the **price** of sharing, and why rarity makes it cheap;
3. **two ideas, one neuron**: the three things a neuron can do, their exact costs, and the **phase diagram**;
4. **three ideas, two neurons**: a few cells of the bigger map;
5. **breaking a pentagon**: make one feature rarer and watch the shape snap;
6. **shapes in a bigger model**: 400 features in 30 neurons, the plateau at one half, and every feature's fraction.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab, CPU only, in about fifteen minutes; nothing to download, no keys.
**One difference from the lecture:** the lecture trained 16,000 models per map, 10 per setting, for 20,000–30,000 steps
on large batches on a GPU. Here the maps are 10 × 10 (every fourth row and column of the lecture's 40 × 40), with fewer
starts and shorter training, so that it all fits on Colab's CPU. The checks say where that matters.

## 0. Setup
Installs (pinned), the plot style, `check()`, and the trainer every section uses.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the small CPU build.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("numpy==2.0.2", "matplotlib==3.10.0", "scipy==1.14.1")

In [ ]:
import math, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch

T0 = time.time()
# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value) + 0.0:.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

**The network** is Lecture 15's: n features squeezed into m neurons and spread back out, x′ = ReLU(Wᵀ W x + b). Each
feature is 0 with probability S, otherwise a number drawn evenly between 0 and 1; its **density** is 1 − S. The loss of a
sample is Σᵢ importanceᵢ × (xᵢ − x′ᵢ)². `train` trains many models at once as one batched tensor, each row with its own
importances and densities, exactly as the lecture's `toy.py` does (Adam, a 5% warm-up, cosine decay to 5%).

In [ ]:
torch.set_num_threads(2)


def train(I, P, n, m, steps, batch=1024, lr=1e-3, seed=0):
    """I, P: (models, n) importance and density of every feature. Returns final W (models, m, n), b, and each model's
    loss on 65,536 fresh samples."""
    g = torch.Generator().manual_seed(seed)
    I = torch.tensor(np.asarray(I), dtype=torch.float32)
    P = torch.tensor(np.asarray(P), dtype=torch.float32)
    R = I.shape[0]
    W = torch.empty(R, m, n)
    for r in range(R):
        torch.nn.init.xavier_normal_(W[r], generator=g)
    W.requires_grad_(True)
    b = torch.zeros(R, n, requires_grad=True)
    opt = torch.optim.Adam([W, b], lr=lr)
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))))

    def sample(B):
        return (torch.rand(R, B, n, generator=g) < P[:, None, :]) * torch.rand(R, B, n, generator=g)

    def out(x):
        return torch.relu(torch.einsum("rbm,rmn->rbn", torch.einsum("rbn,rmn->rbm", x, W), W) + b[:, None, :])

    for step in range(steps):
        x = sample(batch)
        loss = (I[:, None, :] * (x - out(x)) ** 2).sum(-1).mean(-1)
        opt.zero_grad(set_to_none=True); loss.sum().backward(); opt.step(); sched.step()
    with torch.no_grad():
        final = sum((I[:, None, :] * (x - out(x)) ** 2).sum(-1).mean(-1) for x in (sample(8192) for _ in range(8))) / 8
    return W.detach().numpy(), b.detach().numpy(), final.numpy()


def draw(ax, W, title="", keep=0.3):
    """Each feature's arrow in the plane of two neurons."""
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, ls=":", color=INK2))
    ax.axhline(0, color=INK2, lw=0.6); ax.axvline(0, color=INK2, lw=0.6)
    for i in range(W.shape[1]):
        L = np.hypot(*W[:, i])
        if L < 0.05:
            continue
        ax.annotate("", xy=W[:, i], xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK if L > keep else INK2, lw=2))
        ax.text(*(W[:, i] * (1 + 0.2 / L)), str(i + 1), ha="center", va="center", fontsize=10)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)


def gaps(W, keep=0.3):
    """Angles between neighbouring kept arrows, going round the circle, in degrees."""
    nr = np.linalg.norm(W, axis=0)
    ang = np.sort(np.degrees(np.arctan2(W[1], W[0]))[nr > keep])
    return np.diff(np.concatenate([ang, [ang[0] + 360]]))

## 1. The false echo (Plate I)
Lecture 15's regular pentagon: five features of equal importance at S = 0.9 (each on in 1 sample of 10), two neurons.
The lecture trained 10 random starts and kept the best. So do we; this also trains, in the same batch, the 66 models of
Section 5 (breaking a pentagon), which takes a few minutes.

Turn feature 4 on at value 1. The two neurons hold its arrow; each output reads that arrow along its own (a dot product),
adds its bias, and passes the gate.

In [ ]:
D5 = np.array([0.05, 0.036395, 0.026492, 0.019283, 0.014036, 0.010217, 0.007437, 0.005413, 0.00394, 0.002868, 0.0005])                                   # 11 of the lecture's 30 densities for feature 5 (Section 5)
rows5 = [("pentagon", 0.1)] * 10 + [("break", d) for d in D5 for _ in range(6)]
P5 = np.array([[0.1] * 5 if k == "pentagon" else [0.05] * 4 + [d] for k, d in rows5])
t0 = time.time()
W5, b5, L5 = train(np.ones((len(rows5), 5)), P5, n=5, m=2, steps=15_000, seed=1)
print(f"trained {len(rows5)} models in {time.time() - t0:.0f} s")

k = int(np.argmin(L5[:10]))                           # the best of the 10 pentagons
Wp, bp = W5[k], b5[k]
Gp = Wp.T @ Wp                                        # every arrow read along every arrow
a = 3                                                 # feature 4, as on the plate
others = sorted((j for j in range(5) if j != a), key=lambda j: -Gp[a, j])
nb, far = others[:2], others[2:]
print("arrow lengths:", np.round(np.linalg.norm(Wp, axis=0), 3), " biases:", np.round(bp, 3))
print("gaps between neighbours:", np.round(gaps(Wp), 1))
check("its own reading along its arrow", Gp[a, a], 1.21, 0.05)
check("a neighbour's reading (72° away)", float(Gp[a, nb].mean()), 0.373, 0.03)
check("a far one's reading (144° away)", float(Gp[a, far].mean()), -0.979, 0.05)
check("the bias", float(bp.mean()), -0.209, 0.02)
pre = Gp[:, a] + bp
check("its own output: reading + bias", float(pre[a]), 1.001, 0.05)
check("each neighbour's false echo", float(pre[nb].mean()), 0.165, 0.03)
check("the far two, before the gate", float(pre[far].mean()), -1.189, 0.05)
thr = float(-bp[nb].mean() / Gp[a, nb].mean())
check("below this value, even the echo is swallowed", thr, 0.559, 0.05)

fig, axs = plt.subplots(1, 2, figsize=(8.5, 3.4), gridspec_kw=dict(width_ratios=[1, 1.5]))
draw(axs[0], Wp, "the trained pentagon")
v = np.linspace(0, 1.2, 121)
for j, ls in ((a, "-"), (nb[0], "--"), (far[0], ":")):
    axs[1].plot(v, np.maximum(v * Gp[j, a] + bp[j], 0), ls=ls, color=INK, lw=2,
                label={a: "its own output", nb[0]: "a neighbour: the echo", far[0]: "a far one"}[j])
axs[1].axvline(thr, color=INK2, lw=0.8); axs[1].text(thr, 1.3, f" filter: {thr:.2f}", fontsize=9, va="top")
axs[1].set_xlabel("feature 4's value"); axs[1].set_ylabel("output after the gate"); axs[1].legend(fontsize=8)
plt.show()

The extra length of each arrow (about 1.1) exactly pays for the negative bias, so a feature on its own comes back at 1.
The neighbours hear a small false echo; the far two read below zero and the gate clips them. The bias is a **filter**:
for values below about 0.56, even the echo is swallowed.

## 2. The price, and why rarity makes it cheap (Plate II)
The echo costs something only when two neighbours are on at the same time: then each hears the other.

In [ ]:
x2 = np.zeros(5); x2[a] = x2[nb[0]] = 1.0
both = np.maximum(Gp @ x2 + bp, 0)
print("two neighbours on at 1, outputs:", np.round(both, 3))
check("each of the two reads", float(both[[a, nb[0]]].mean()), 1.375, 0.05)

rng = np.random.default_rng(16)
def collisions(p, n_samples):
    on = rng.random((n_samples, 5)) < p
    ring = [(i, (i + 1) % 5) for i in range(5)]        # the five neighbouring pairs of a pentagon
    return on.sum(), sum(int((on[:, i] & on[:, j]).sum()) for i, j in ring)
big = 1_000_000
shows, hits = collisions(0.1, big)
print(f"per 100 samples at density 0.1: features show up {shows / big * 100:.1f} times, neighbours collide {hits / big * 100:.2f} times")
check("per 100 samples at density 0.1: appearances", shows / big * 100, 50, 0.5)
check("per 100 samples at density 0.1: neighbour collisions", hits / big * 100, 5, 0.1)

ps = np.logspace(-2, np.log10(0.5), 30)
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.loglog(ps, ps, color=INK, lw=2, label="gain: a feature appears (p)")
ax.loglog(ps, ps ** 2, color=INK, lw=2, ls="--", label="price: two neighbours collide (p × p)")
ax.set_xlabel("density p"); ax.set_ylabel("share of samples"); ax.legend(fontsize=8); plt.show()

The gain grows like p and the price like p × p, so the rarer the features, the cheaper sharing becomes. At p = 0.5,
collisions are everywhere.

## 3. Two ideas, one neuron (Plates III–IV)
The smallest squeeze: two features, one neuron. Feature 1 has importance 1, feature 2 has importance r, and both have
density p. The paper's three candidates:

- (a) keep feature 1: W = [1, 0]. Feature 2 is dropped, and its best guess is its average;
- (b) keep feature 2: W = [0, 1];
- (c) keep both, pointing opposite ways: W = [1, −1], with the two biases chosen as well as possible.

Their expected losses can be computed exactly (a dropped feature costs its variance, r × (p/3 − p²/4); the third by
quadrature on a 64 × 64 grid and a search over the two biases), as the lecture did.

In [ ]:
from scipy.optimize import minimize

q = (np.arange(64) + 0.5) / 64                         # midpoint nodes on [0, 1]
X1, X2 = np.meshgrid(q, q, indexing="ij")


def drop_loss(p):
    return p / 3 - p * p / 4


def antipodal_loss(params, r, p):
    b1, b2 = params
    L = lambda x1, x2: (x1 - np.maximum(x1 - x2 + b1, 0)) ** 2 + r * (x2 - np.maximum(x2 - x1 + b2, 0)) ** 2
    return (1 - p) ** 2 * L(0.0, 0.0) + p * (1 - p) * (L(q, 0.0).mean() + L(0.0, q).mean()) + p * p * L(X1, X2).mean()


def three_losses(r, p):
    best = min((minimize(antipodal_loss, s, args=(r, p), method="Nelder-Mead",
                         options=dict(maxiter=400, xatol=1e-6, fatol=1e-10)) for s in ([0.0, 0.0], [-0.2, -0.2])),
               key=lambda o: o.fun)
    return np.array([r * drop_loss(p), drop_loss(p), best.fun])


DENS, REL = np.logspace(-2, 0, 40), np.logspace(-1, 1, 40)   # the lecture's grid
NAMES = ["keep feature 1", "keep feature 2", "both, opposite ways"]
for (i, j, want), lec in zip([[0, 39, "keep feature 1"], [39, 39, "keep feature 2"], [19, 10, "both, opposite ways"]], [0.008333, 0.083333, 0.000343]):
    L3 = three_losses(REL[i], DENS[j])
    print(f"r = {REL[i]:.2f}, p = {DENS[j]:.3f}: losses {np.round(L3, 5)} -> {NAMES[int(np.argmin(L3))]}")
    check(f"r = {REL[i]:.2f}, p = {DENS[j]:.3f}: the cheapest", NAMES[int(np.argmin(L3))], want)
    check(f"r = {REL[i]:.2f}, p = {DENS[j]:.3f}: its loss", float(L3.min()), lec, 1e-5)

**The phase diagram.** Do it for every setting and mark the cheapest: · = feature 2 not represented, D = feature 2 takes
the neuron (a dedicated dimension), S = superposition. Three rows of the lecture's 40 × 40 grid give its borders.

In [ ]:
t0 = time.time()
theory_rows = {}
for i in (0, 19, 39):
    theory_rows[i] = np.array([int(np.argmin(three_losses(REL[i], d))) for d in DENS])
    sup = DENS[theory_rows[i] == 2]
    print(f"r = {REL[i]:5.2f}: " + "".join(".DS"[g] for g in [{0: 0, 1: 1, 2: 2}[k] for k in theory_rows[i]]),
          f" superposition up to density {sup.max():.3f}")
check("theory: superposition up to density, at r = 0.1", float(DENS[theory_rows[0] == 2].max()), 0.1512, 1e-3)
check("theory: superposition up to density, at r = 0.94", float(DENS[theory_rows[19] == 2].max()), 0.5541, 1e-3)
check("theory: superposition up to density, at r = 10", float(DENS[theory_rows[39] == 2].max()), 0.1512, 1e-3)
SUB = [2, 6, 10, 14, 18, 22, 26, 30, 34, 38]                                           # every 4th row and column of the 40 x 40 grid
theory_sub = np.array([[int(np.argmin(three_losses(REL[i], DENS[j]))) for j in SUB] for i in SUB])
print(f"theory for {len(SUB) ** 2} cells in {time.time() - t0:.0f} s")
LECTURE_THEORY = np.array([[2, 2, 2, 2, 2, 2, 0, 0, 0, 0], [2, 2, 2, 2, 2, 2, 2, 0, 0, 0], [2, 2, 2, 2, 2, 2, 2, 2, 0, 0], [2, 2, 2, 2, 2, 2, 2, 2, 0, 0], [2, 2, 2, 2, 2, 2, 2, 2, 2, 0], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1], [2, 2, 2, 2, 2, 2, 1, 1, 1, 1]])                    # the lecture's theory map at these cells
check("theory, 10 x 10 cells: cells that match the lecture's theory", int((theory_sub == LECTURE_THEORY).sum()), 100)

**Now train it.** For each of the 100 cells, 5 models from random starts (5,000 steps on batches of 512; about two
minutes). As the lecture (and the paper) did, drop the
highest-loss model in each cell and take the majority of the rest. A model's feature 2 is *not represented* if its
weight is under 0.5 in size, *dedicated* if it barely overlaps feature 1 (squared overlap under 0.1), and in
*superposition* otherwise.

In [ ]:
rows2 = [(REL[i], DENS[j]) for i in SUB for j in SUB for _ in range(5)]
t0 = time.time()
W2, b2, L2 = train(np.array([[1.0, r] for r, d in rows2]), np.array([[d, d] for r, d in rows2]), n=2, m=1,
                   steps=5_000, batch=512, lr=1e-2, seed=2)
print(f"trained {len(rows2)} models in {time.time() - t0:.0f} s")


def phase2(Wk):
    nrm = np.abs(Wk[0])
    if nrm[1] < 0.5:
        return 0
    return 1 if (np.sign(Wk[0, 1]) * Wk[0, 0]) ** 2 < 0.1 else 2


trained = np.zeros((len(SUB), len(SUB)), int)
for c in range(len(SUB) ** 2):
    idx = np.arange(c * 5, (c + 1) * 5)
    keep = idx[np.argsort(L2[idx])[:-1]]                 # drop the worst
    v, cnt = np.unique([phase2(W2[k]) for k in keep], return_counts=True)
    trained.flat[c] = v[np.argmax(cnt)]
agree = float((trained == theory_sub).mean())
print("trained (left) and theory (right); rows: r from 10 (top) to 0.13; columns: density 0.012 -> 0.79")
for i in range(len(SUB) - 1, -1, -1):
    print(f"r = {REL[SUB[i]]:5.2f}  " + "".join(".DS"[g] for g in trained[i]) + "   " + "".join(".DS"[g] for g in theory_sub[i]))
print(f"cells where training agrees with theory: {agree:.0%} (the lecture's 16,000 models, on these same cells: 86%)")
check("cells where trained and theory agree", agree, 0.86, 0.06)

fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, M, t in ((axs[0], theory_sub, "theory"), (axs[1], trained, "trained")):
    ax.imshow(M, origin="lower", cmap=LinearSegmentedColormap.from_list("ph", [PAPER, INK2, INK]), vmin=0, vmax=2)
    ax.set_xticks([0, 9]); ax.set_xticklabels([f"{DENS[SUB[0]]:.2f}", f"{DENS[SUB[-1]]:.2f}"])
    ax.set_yticks([0, 9]); ax.set_yticklabels([f"{REL[SUB[0]]:.2f}", f"{REL[SUB[-1]]:.1f}"])
    ax.set_xlabel("density"); ax.set_ylabel("importance of feature 2"); ax.set_title(t, fontsize=10)
fig.suptitle("black: superposition · grey: feature 2 takes the neuron · paper: not represented", fontsize=9)
plt.show()

## 4. Three ideas, two neurons (Plate V)
Two neurons, and feature 3's importance r varies; all three features share the density. Five ways the three can sit:
**·** feature 3 dropped; **A** feature 3 opposite another; **P** feature 3 alone, the other two opposite each other;
**D** feature 3 alone, one other dropped; **T** a triangle. We train four cells of the lecture's map, 10 models each.

In [ ]:
CELLS3 = [[1, 13], [1, 39], [34, 9], [34, 39]]                                       # (row, column) of the lecture's 40 x 40 grid
rows3 = [(REL[i], DENS[j]) for i, j in CELLS3 for _ in range(10)]
t0 = time.time()
W3, b3, L3s = train(np.array([[1.0, 1.0, r] for r, d in rows3]), np.array([[d] * 3 for r, d in rows3]), n=3, m=2,
                    steps=10_000, lr=1e-2, seed=3)
print(f"trained {len(rows3)} models in {time.time() - t0:.0f} s")


def kind3(Wk):
    """The lecture's rule (analyze_phase3.py)."""
    nrm = np.linalg.norm(Wk, axis=0)
    rep = nrm > 0.5
    if not rep[2]:
        return "·"
    u = Wk / np.maximum(nrm, 1e-9)
    cos = u.T @ u
    if rep[0] and rep[1] and min(cos[0, 1], cos[0, 2], cos[1, 2]) > -0.75 and max(cos[0, 1], cos[0, 2], cos[1, 2]) < -0.25:
        return "T"
    if any(rep[j] and cos[2, j] < -0.75 for j in (0, 1)):
        return "A"
    if rep[0] and rep[1] and cos[0, 1] < -0.75:
        return "P"
    return "D"


fig, axs = plt.subplots(1, 4, figsize=(11, 2.9))
for c, ((i, j), want, ax) in enumerate(zip(CELLS3, ["A", "\u00b7", "P", "D"], axs)):
    idx = np.arange(c * 10, (c + 1) * 10)
    keep = idx[np.argsort(L3s[idx])[:-1]]
    kinds = [kind3(W3[k]) for k in keep]
    common = max(set(kinds), key=kinds.count)
    print(f"r = {REL[i]:.2f}, density {DENS[j]:.3f}: {''.join(kinds)}  -> {common}")
    check(f"r = {REL[i]:.2f}, density {DENS[j]:.3f}: the commonest", common, want)
    draw(ax, W3[idx[np.argmin(L3s[idx])]], f"r = {REL[i]:.2f}, p = {DENS[j]:.3f}: {common}", keep=0.5)
plt.show()

Below importance 1, feature 3 pairs off opposite another feature when rare, and is dropped when common. Above 1, it
stands alone: the other two share the second neuron when rare, or one of them is dropped when common. The lecture's map
also shows that above 1 the models often settle for the second-best layout; you may see that in the letters above.

## 5. Breaking a pentagon (Plate VI)
Five equal features, two neurons, all at density 0.05: a regular pentagon. Now make feature 5 alone rarer, density 0.05 →
0.0005. These models were trained in Section 1 (6 starts per density, at 11 of the lecture's 30 densities: every other
one down to 0.0054, then the last). For each density we keep the lowest-loss model.

In [ ]:
brk = np.arange(10, len(rows5))
best5, n5kept = [], []
for d in D5:
    idx = brk[np.isclose([rows5[k][1] for k in brk], d)]
    best5.append(W5[idx[np.argmin(L5[idx])]])
    n5kept.append(int((np.linalg.norm(W5[idx][:, :, 4], axis=1) > 0.3).sum()))
len5 = np.array([np.linalg.norm(Wb[:, 4]) for Wb in best5])
for d, Wb, kk in zip(D5, best5, n5kept):
    print(f"density {d:.4f}: lengths {np.round(np.linalg.norm(Wb, axis=0), 2)}  gaps {np.round(gaps(Wb), 0)}  feature 5 kept by {kk} of 6")
check("density 0.05: features kept", int((np.linalg.norm(best5[0], axis=0) > 0.3).sum()), 5)
check("density 0.05: a regular pentagon (largest gap error from 72°)", float(np.abs(gaps(best5[0]) - 72).max()), 0.0, 3)
check("density 0.05: arrow length", float(np.linalg.norm(best5[0], axis=0).mean()), 1.135, 0.03)
check("density 0.0005: feature 5 dropped", bool(len5[-1] < 0.3), True)
check("density 0.0005: the other four at right angles (largest gap error from 90°)", float(np.abs(gaps(best5[-1]) - 90).max()), 0.0, 3)
snap = int(np.argmax(len5 < 0.3))                     # the first density whose best model has dropped feature 5
print(f"the lowest-loss model switches between densities {D5[snap - 1]:.4f} and {D5[snap]:.4f} "
      f"(the lecture: {0.014036081019705883:.4f} and {0.011975133099937429:.4f})")
check("feature 5 grows before it goes (largest length / its length at 0.05)", float(len5[:snap].max() / len5[0]), 1.028, 0.02)
check("the switch lies where the lecture found both shapes among its starts (density)", bool(0.00634 <= D5[snap] <= 0.04266), True)

fig, axs = plt.subplots(1, 4, figsize=(11, 2.9))
for ax, k in zip(axs, (0, snap - 2, snap - 1, snap)):
    draw(ax, best5[k], f"feature 5 at {D5[k]:.4f}")
plt.show()

No in-between: feature 5 grows a little, its neighbours lean in, and then it is gone and the other four sit as two
opposite pairs. Two solutions coexist over a range of densities, and which one is cheaper switches at once: a
first-order phase change.

## 6. Shapes in a bigger model (Plates VII–IX)
**A feature's share of a dimension.** For feature i, Dᵢ = ‖Wᵢ‖² ÷ Σⱼ (Ŵᵢ · Wⱼ)², where Ŵᵢ is its arrow scaled to length
1: its own length, divided by how much every arrow (itself included) lies along it. A feature alone in its dimension gets
1. For k arrows spread evenly through d dimensions, every Dᵢ is d ÷ k. First on the ideal shapes:

In [ ]:
def featdim(W):
    nrm = np.linalg.norm(W, axis=0)
    Gh = (W / np.maximum(nrm, 1e-9)).T @ W
    return nrm ** 2 / np.maximum((Gh ** 2).sum(1), 1e-12), nrm


poly = lambda k: np.stack([np.cos(2 * np.pi * np.arange(k) / k), np.sin(2 * np.pi * np.arange(k) / k)])
tetra = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]).T / np.sqrt(3)
t8 = np.arange(8) * np.pi / 4                           # two squares, 45° apart, above and below: cos² of the tilt = 1/3
antiprism = np.stack([np.sqrt(2 / 3) * np.cos(t8), np.sqrt(2 / 3) * np.sin(t8), np.where(np.arange(8) % 2, 1, -1) / np.sqrt(3)])
IDEAL = {"digon": (np.array([[1.0, -1.0]]), 1 / 2), "triangle": (poly(3), 2 / 3), "pentagon": (poly(5), 2 / 5),
         "tetrahedron": (tetra, 3 / 4), "square antiprism": (antiprism, 3 / 8)}
for name, (Wsh, frac) in IDEAL.items():
    Dv, _ = featdim(Wsh)
    print(f"{name:17s} {Wsh.shape[1]} arrows in {Wsh.shape[0]} dimensions: D = {Dv.mean():.4f}")
    check(f"{name}: D", float(Dv.mean()), frac, 1e-9)

**The bigger model.** 400 features of equal importance squeezed into 30 neurons, at two sparsities, 1/(1 − S) = 2.40
and 5.00 (the lecture swept 40). One model each, 15,000 steps; it takes about four minutes. Then two measures:

- **dimensions per feature**, 30 ÷ ‖W‖²: how much room the average feature gets;
- each learned feature's **D**, and how many sit within 0.02 of one of the shapes' fractions.

In [ ]:
INV = np.array([2.4049, 4.9966])                                 # 1 / (1 - S)
t0 = time.time()
W400, b400, L400 = train(np.ones((len(INV), 400)), np.repeat((1 / INV)[:, None], 400, 1), n=400, m=30, steps=15_000, seed=4)
print(f"trained {len(INV)} models of 400 features in {time.time() - t0:.0f} s")
FR = {"3/4 tetrahedron": 3 / 4, "2/3 triangle": 2 / 3, "1/2 digon": 1 / 2, "2/5 pentagon": 2 / 5, "3/8 square antiprism": 3 / 8}
near, dpf = [], []
fig, ax = plt.subplots(figsize=(7, 3))
for k, inv in enumerate(INV):
    Dv, nrm = featdim(W400[k])
    learned = nrm > 0.5
    dpf.append(30 / float((W400[k] ** 2).sum()))
    near.append({f: int((np.abs(Dv[learned] - v) < 0.02).sum()) for f, v in FR.items()})
    print(f"1/(1-S) = {inv:.2f}: dimensions per feature {dpf[-1]:.3f}, learned {learned.sum()}, near a fraction: {near[-1]}")
    ax.scatter(np.full(learned.sum(), k) + np.random.default_rng(k).uniform(-0.15, 0.15, learned.sum()), Dv[learned], s=6, color=INK)
for f, v in FR.items():
    ax.axhline(v, color=INK2, lw=0.6, ls=":"); ax.text(len(INV) - 0.65, v, f, fontsize=8, va="center")
ax.set_xticks(range(len(INV))); ax.set_xticklabels([f"1/(1−S) = {v:.2f}" for v in INV]); ax.set_ylabel("each learned feature's D")
ax.set_xlim(-0.5, len(INV) + 0.4); plt.show()

In [ ]:
check("1/(1-S) = 2.40: dimensions per feature (the plateau at one half)", dpf[0], 0.51, 0.02)
check("1/(1-S) = 2.40: the commonest fraction", max(near[0], key=near[0].get), "1/2 digon")
check("1/(1-S) = 2.40: learned features on 1/2", near[0]["1/2 digon"], 58, 8)
check("1/(1-S) = 5.00: dimensions per feature", dpf[1], 0.363, 0.02)
# the lecture's two seeds gave 38 + 70 = 108 and 42 + 42 = 84 here: its best seed is the expected value, the other's distance the tolerance
check("1/(1-S) = 5.00: learned features on 2/5 or 3/8", near[1]["2/5 pentagon"] + near[1]["3/8 square antiprism"], 84, 24)
check("1/(1-S) = 5.00: none on 1/2 any more", near[1]["1/2 digon"], 0)

At 1/(1 − S) = 2.4 the average feature gets half a dimension: every learned feature has an opposite partner (the
lecture's "sticky plateau"). Rarer still, the shares drop to 2/5 and 3/8. A share is not yet a shape, though: at 5.00
most of the features near 2/5 are opposite pairs leaning a little on others, and the clean pentagons and square antiprism
sit nearer 4.3 (Plate IX). **Find a digon:** two learned features with D near 1/2 that point exactly opposite ways.

In [ ]:
Wd = W400[0]
Dv, nrm = featdim(Wd)
i = int(np.where((nrm > 0.5) & (np.abs(Dv - 0.5) < 0.02))[0][0])
cos = (Wd[:, i] / nrm[i]) @ (Wd / np.maximum(nrm, 1e-9))
j = int(np.argmin(cos))
print(f"feature {i} (D = {Dv[i]:.3f}) and its partner, feature {j} (D = {Dv[j]:.3f}): cos = {cos[j]:.3f}")
check("a digon: the partner points the opposite way (cos)", float(cos[j]), -1.0, 0.02)

## Your turn
Each exercise has a one-line hint; worked solutions are at the end.

1. **The filter.** In the trained pentagon, feature 4 is on at 0.5. Does its neighbour hear an echo? *Hint:* `Gp`, `bp`.
2. **Gain and price.** At density p, how often do two given neighbours collide, per appearance of one feature? What is it
   at p = 0.1 and at p = 0.5? *Hint:* `collisions`.
3. **Another border.** Where does superposition stop, in theory, for feature 2 worth 3.07 × feature 1 (row 29 of the
   grid)? *Hint:* `three_losses`.
4. **Every one a digon?** In the 1/(1 − S) = 2.40 model, does *every* learned feature with D near 1/2 have a partner
   pointing exactly the other way? *Hint:* for each, the most negative cosine with any other arrow.
5. **Where did feature 5 go?** In Section 5, at the first density where the best model drops feature 5, how many of the
   starts still kept it? *Hint:* `n5kept`.

In [ ]:
# TODO 1

In [ ]:
# TODO 2

In [ ]:
# TODO 3

In [ ]:
# TODO 4

In [ ]:
# TODO 5

## Solutions

In [ ]:
# 1. No: the neighbour reads 0.5 x its overlap, less than the bias, so the gate passes nothing.
x = np.zeros(5); x[a] = 0.5
outs = np.maximum(Gp @ x + bp, 0)
print(f"neighbour reads {0.5 * Gp[a, nb[0]]:.3f} against a bias of {bp[nb[0]]:.3f}; its output: {outs[nb[0]]:.3f}")
check("the neighbour's output at value 0.5", float(outs[nb[0]]), 0.0, 1e-9)

In [ ]:
# 2. Collisions per appearance grow like p: 1 in 10 at p = 0.1 (rare enough to pay), 1 in 2 at p = 0.5.
for p in (0.1, 0.5):
    s, h = collisions(p, 400_000)
    print(f"p = {p}: neighbour collisions per appearance {h / s:.3f}")
    check(f"p = {p}: collisions per appearance (two neighbours per feature, each on with p)", h / s, p, 0.01)

In [ ]:
# 3. From the theory row at r = 3.07.
row = np.array([int(np.argmin(three_losses(REL[29], d))) for d in DENS])
print("r = 3.07: " + "".join(".DS"[g] for g in row), f" superposition up to density {DENS[row == 2].max():.3f}")
check("theory at r = 3.07: superposition up to density", float(DENS[row == 2].max()), 0.3455, 1e-3)

In [ ]:
# 4. On the plateau every learned feature on 1/2 has a partner pointing exactly the other way.
Wd = W400[0]
Dv, nrm = featdim(Wd)
half = np.where((nrm > 0.5) & (np.abs(Dv - 0.5) < 0.02))[0]
U = Wd / np.maximum(nrm, 1e-9)
C = U[:, half].T @ U                                  # each half-feature's cosine with every feature
partner = np.argmin(C, axis=1)
best_cos = C[np.arange(len(half)), partner]
print(f"{len(half)} features on 1/2; their partners' cosines range from {best_cos.min():.3f} to {best_cos.max():.3f}")
print("is each partner itself on 1/2, and paired back?", all(p_ in half and int(np.argmin(U[:, p_] @ U)) == h for h, p_ in zip(half, partner)))
check("share of the 1/2 features with an exactly opposite partner (cos below -0.98)", float((best_cos < -0.98).mean()), 1.0, 0.0)

In [ ]:
# 5. The switch is between coexisting solutions: some starts still keep feature 5 where the best one has dropped it.
print("starts that keep feature 5, by density:", dict(zip(np.round(D5, 4).tolist(), n5kept)))
print(f"at {D5[snap]:.4f}: {n5kept[snap]} of 6 starts still keep it")

In [ ]:
print(f"notebook finished in {(time.time() - T0) / 60:.1f} min")